In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("../data/mash_dist.tsv", sep="\t", header=None)
df.columns = ["s1", "s2", "dist", "p", "shared"]
df["s1"] = df["s1"].str.replace("_R1.fq.gz", "", regex=False)
df["s2"] = df["s2"].str.replace("_R1.fq.gz", "", regex=False)
matrix = df.pivot(index="s1", columns="s2", values="dist")
X = matrix.values

In [ ]:
meta = pd.DataFrame({"sample": matrix.index})
meta["status"] = meta["sample"].apply(lambda x: "healthy" if x.startswith("ERR") else "disease")
meta = meta.set_index("sample")

In [ ]:
from modules.pca import plot_pca

pca_df, pca = plot_pca(X, meta, method="Mash")

Building a hierarchical tree

In [ ]:
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram

dist_array = squareform(matrix.values)
Z = linkage(dist_array, method='complete')

meta_dict = meta['status'].to_dict()
labels = matrix.index
colors = ['green' if meta_dict[s]=='healthy' else 'red' for s in labels]

plt.figure(figsize=(10, 6))
dendrogram(Z, labels=labels)

ax = plt.gca()
xlbls = ax.get_xmajorticklabels()

for lbl in xlbls:
    lbl.set_color('green' if meta_dict[lbl.get_text()]=='healthy' else 'red')

plt.title("Mash distance dendrogram")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()